In [1]:
import pandas as pd
import numpy as np
import shap
from sklearn.ensemble import GradientBoostingClassifier

In [2]:
# --- STEP 1: DATA & MODEL SETUP (RECAP FROM PHASE 4) ---
df = pd.read_csv('financial_inclusion_eda_final.csv')
X = df.drop(['Applicant_ID', 'Default_Status'], axis=1)
y = df['Default_Status']

# Train the engine we validated with a 0.55 Gini
model = GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42)
model.fit(X, y)

GradientBoostingClassifier(max_depth=4, random_state=42)

In [3]:
# --- STEP 2: CALCULATE SHAP VALUES ---
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X)

In [4]:
# --- STEP 3: MAPPING TECHNICAL FEATURES TO "HUMAN REASONS" ---
# This ensures we follow NCA Section 62: Explainable Decisions
reason_map = {
    'Sim_Tenure_Months': 'Insufficient Digital Stability (Short SIM Tenure)',
    'Volatility_Index': 'High Spending Volatility Detected',
    'Electricity_Payment_Gap_Days': 'Inconsistent Utility Payment History',
    'Airtime_Consistency_Score': 'Irregular Airtime Recharge Patterns',
    'Avg_Wallet_Inflow': 'Low Financial Velocity (Monthly Inflow)',
    'Data_Voice_Ratio': 'Low Digital Literacy Proxy'
}

In [5]:
# --- STEP 4: BATCH EXTRACTION OF REASON CODES ---
reason_list = []

# We focus on the first 100 applicants to keep the output clean
for i in range(100):
    # Identify features with the highest positive SHAP values (pushing risk UP)
    # If the model is binary, shap_values[i] gives the impact for the 'Default' class
    sv = shap_values[i]
    
    # Sort indices by impact (descending)
    top_indices = np.argsort(sv)[::-1][:2] # Get Top 2 reasons
    
    reasons = [reason_map[X.columns[idx]] for idx in top_indices if sv[idx] > 0]
    
    reason_list.append({
        'Applicant_ID': df.iloc[i]['Applicant_ID'],
        'Default_Probability': round(model.predict_proba(X.iloc[[i]])[0][1], 2),
        'Primary_Reason': reasons[0] if len(reasons) > 0 else 'N/A (Low Risk)',
        'Secondary_Reason': reasons[1] if len(reasons) > 1 else 'N/A'
    })

In [6]:
# --- STEP 5: THE FINAL OPERATIONAL TABLE ---
reason_df = pd.DataFrame(reason_list)
print("--- NCA COMPLIANT REASON CODE TABLE ---")
print(reason_df.head(10))

--- NCA COMPLIANT REASON CODE TABLE ---
   Applicant_ID  Default_Probability  \
0        1001.0                 0.32   
1        1002.0                 0.35   
2        1003.0                 0.06   
3        1004.0                 0.59   
4        1005.0                 0.18   
5        1006.0                 0.19   
6        1007.0                 0.76   
7        1008.0                 0.82   
8        1009.0                 0.31   
9        1010.0                 0.46   

                                      Primary_Reason  \
0               Inconsistent Utility Payment History   
1            Low Financial Velocity (Monthly Inflow)   
2                                     N/A (Low Risk)   
3  Insufficient Digital Stability (Short SIM Tenure)   
4                  High Spending Volatility Detected   
5                                     N/A (Low Risk)   
6                  High Spending Volatility Detected   
7  Insufficient Digital Stability (Short SIM Tenure)   
8            Lo